# CekAjaYuk - TensorFlow Deep Learning Training
## Training CNN untuk Deteksi Iklan Lowongan Kerja Palsu

Notebook ini berisi proses training Deep Learning model menggunakan TensorFlow:
1. Import libraries dan setup
2. Data preparation untuk CNN
3. Model architecture design
4. Transfer learning dengan pre-trained models
5. Training dan validation
6. Model evaluation
7. Save trained model

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Flatten, Dropout, 
    GlobalAveragePooling2D, BatchNormalization, Activation
)
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
)
from tensorflow.keras.utils import plot_model

# Sklearn for metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Configuration
RANDOM_STATE = 42
IMG_SIZE = (224, 224, 3)
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2

# Paths
DATA_DIR = '../data'
MODELS_DIR = '../models'
LOGS_DIR = '../logs'

# Create directories
os.makedirs(LOGS_DIR, exist_ok=True)

# Set random seeds
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")

In [ ]:
# Create synthetic image data for demonstration
def create_synthetic_image_data():
    """
    Create synthetic image data for demonstration purposes
    In real implementation, you would load actual job poster images
    """
    print("Creating synthetic image data...")
    
    # Generate synthetic images
    num_samples = 1000
    
    # Create synthetic images with different patterns for genuine vs fake
    X_data = []
    y_data = []
    
    for i in range(num_samples):
        # Create base image
        img = np.random.rand(224, 224, 3)
        
        if i < num_samples // 2:  # Genuine jobs
            # Add patterns that might indicate genuine job posts
            # More structured layout (add some geometric patterns)
            img[50:150, 50:150] = np.random.rand(100, 100, 3) * 0.8 + 0.2
            # Add "logo" area
            img[10:50, 10:50] = np.random.rand(40, 40, 3) * 0.6 + 0.4
            label = 1  # Genuine
        else:  # Fake jobs
            # Add patterns that might indicate fake job posts
            # More chaotic, less structured
            noise = np.random.rand(224, 224, 3) * 0.5
            img = img * 0.7 + noise * 0.3
            label = 0  # Fake
        
        X_data.append(img)
        y_data.append(label)
    
    X_data = np.array(X_data)
    y_data = np.array(y_data)
    
    print(f"Created {len(X_data)} synthetic images")
    print(f"Image shape: {X_data[0].shape}")
    print(f"Label distribution: {np.bincount(y_data)}")
    
    return X_data, y_data

# Create synthetic data
X_images, y_labels = create_synthetic_image_data()

In [ ]:
# Split data
from sklearn.model_selection import train_test_split

# Split into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(
    X_images, y_labels, test_size=0.3, random_state=RANDOM_STATE, stratify=y_labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Data augmentation
def create_data_generators():
    """
    Create data generators with augmentation
    """
    train_datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    
    val_datagen = ImageDataGenerator()  # No augmentation for validation
    
    return train_datagen, val_datagen

train_datagen, val_datagen = create_data_generators()
print("Data generators created!")

In [ ]:
# Custom CNN Model
def create_custom_cnn():
    """
    Create a custom CNN architecture
    """
    model = Sequential([
        # First Convolutional Block
        Conv2D(32, (3, 3), activation='relu', input_shape=IMG_SIZE),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Dropout(0.25),
        
        # Second Convolutional Block
        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Dropout(0.25),
        
        # Third Convolutional Block
        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Dropout(0.25),
        
        # Classifier
        GlobalAveragePooling2D(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    
    return model

# Create custom CNN
custom_cnn = create_custom_cnn()
custom_cnn.summary()

In [ ]:
# Transfer Learning with VGG16
def create_transfer_learning_model():
    """
    Create transfer learning model using VGG16
    """
    # Load pre-trained VGG16 model
    base_model = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=IMG_SIZE
    )
    
    # Freeze base model layers
    base_model.trainable = False
    
    # Add custom classification head
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    
    return model

# Create transfer learning model
transfer_model = create_transfer_learning_model()
transfer_model.summary()

In [ ]:
# Compile models
def compile_model(model, learning_rate=LEARNING_RATE):
    """
    Compile model with optimizer and loss function
    """
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )
    return model

# Compile both models
custom_cnn = compile_model(custom_cnn)
transfer_model = compile_model(transfer_model)

print("Models compiled successfully!")

In [ ]:
# Setup callbacks
def create_callbacks(model_name):
    """
    Create training callbacks
    """
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=os.path.join(MODELS_DIR, f'{model_name}_best.h5'),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),
        TensorBoard(
            log_dir=os.path.join(LOGS_DIR, model_name),
            histogram_freq=1
        )
    ]
    
    return callbacks

# Create callbacks for both models
custom_callbacks = create_callbacks('custom_cnn')
transfer_callbacks = create_callbacks('transfer_learning')

print("Callbacks created successfully!")

In [ ]:
# Train Custom CNN
print("Training Custom CNN...")

custom_history = custom_cnn.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=custom_callbacks,
    verbose=1
)

print("Custom CNN training completed!")

In [ ]:
# Train Transfer Learning Model
print("Training Transfer Learning Model...")

transfer_history = transfer_model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=transfer_callbacks,
    verbose=1
)

print("Transfer learning model training completed!")

In [ ]:
# Evaluate models
def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate model performance
    """
    print(f"\nEvaluating {model_name}...")
    
    # Predictions
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    # Calculate metrics
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"AUC-ROC: {auc:.4f}")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }

# Evaluate both models
custom_results = evaluate_model(custom_cnn, X_test, y_test, "Custom CNN")
transfer_results = evaluate_model(transfer_model, X_test, y_test, "Transfer Learning")

In [ ]:
# Save the best model
def save_best_model():
    """
    Save the best performing model
    """
    # Compare models
    if custom_results['f1'] > transfer_results['f1']:
        best_model = custom_cnn
        best_name = 'custom_cnn'
        best_results = custom_results
    else:
        best_model = transfer_model
        best_name = 'transfer_learning'
        best_results = transfer_results
    
    # Save best model
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_path = os.path.join(MODELS_DIR, f'tensorflow_model_{timestamp}.h5')
    best_model.save(model_path)
    
    # Save latest model (overwrite)
    latest_path = os.path.join(MODELS_DIR, 'tensorflow_model_latest.h5')
    best_model.save(latest_path)
    
    # Save model metadata
    metadata = {
        'model_type': best_name,
        'timestamp': timestamp,
        'test_accuracy': best_results['accuracy'],
        'test_f1': best_results['f1'],
        'test_auc': best_results['auc'],
        'input_shape': IMG_SIZE,
        'training_epochs': EPOCHS,
        'batch_size': BATCH_SIZE
    }
    
    import json
    metadata_path = os.path.join(MODELS_DIR, f'tensorflow_metadata_{timestamp}.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)
    
    print(f"\nBest model saved: {model_path}")
    print(f"Latest model: {latest_path}")
    print(f"Best model: {best_name}")
    print(f"Best F1-Score: {best_results['f1']:.4f}")
    
    return best_model, best_results

# Save the best model
best_model, best_results = save_best_model()

print("\n=== TensorFlow Training Completed ===")
print(f"Best Model Performance:")
print(f"- Test Accuracy: {best_results['accuracy']:.4f}")
print(f"- Test F1-Score: {best_results['f1']:.4f}")
print(f"- Test AUC-ROC: {best_results['auc']:.4f}")